<a href="https://colab.research.google.com/github/AnnesaAD05/Python-Projects-Informatics-1st-year-annesa/blob/main/%F0%9F%93%B0_Personalized_News_Digest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Personalized News Digest with NewsAPI

We will now integrate with NewsAPI to fetch real news articles.

### NewsAPI Setup

First, we need to install the `newsapi-python` library and initialize the NewsAPI client with your `NEWS_API_KEY`. Make sure your `NEWS_API_KEY` is stored as a Colab secret.

In [25]:
import sys

# Install the NewsAPI Python client library
!{sys.executable} -m pip install newsapi-python

In [26]:
from newsapi import NewsApiClient
from google.colab import userdata

# Initialize NewsAPI client
NEWS_API_KEY = userdata.get('NEWS_API_KEY')
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

### Fetching News from NewsAPI

Now, let's define the function `fetch_news_from_api` that calls the NewsAPI. This function will fetch articles based on a query and return them in a structured format. I've added a print statement to help debug if no news is returned.

In [27]:
def fetch_news_from_api(q=None, language='en', country='us', category=None, page_size=20):
    """Fetches news articles from NewsAPI based on query parameters."""
    try:
        if q: # If a query is provided, use everything to search for articles
            articles_response = newsapi.get_everything(q=q, language=language, sort_by='relevancy', page_size=page_size)
        else: # Otherwise, get top headlines based on category/country
            articles_response = newsapi.get_top_headlines(category=category, language=language, country=country, page_size=page_size)

        if articles_response['status'] == 'ok':
            print(f"Successfully fetched {len(articles_response['articles'])} articles from NewsAPI for query: '{q}'")
        else:
            print(f"NewsAPI returned status: {articles_response['status']}. Message: {articles_response.get('message', 'No message provided.')}")
            return []

        parsed_articles = []
        for article in articles_response['articles']:
            if article['title'] and article['description'] and article['url']:
                parsed_articles.append({
                    'title': article['title'],
                    'description': article['description'],
                    'category': category if category else 'general', # Assign category if available
                    'link': article['url']
                })
        return parsed_articles
    except Exception as e:
        print(f"Error fetching news from API: {e}")
        return []

### Personalized News Digest Function

Now, let's redefine the `get_personalized_news` function to use `fetch_news_from_api`. This function will filter the articles and format them for display.

In [28]:
def get_personalized_news(keyword, top_n=None):
    """Filters news articles by keyword and formats them for display using NewsAPI."""
    # Use the real API to fetch news
    all_news = fetch_news_from_api(q=keyword, page_size=20) # Fetch more articles to ensure enough for filtering

    # The API call itself (q=keyword) already performs a significant part of the filtering.
    # We'll keep an additional check for robustness if the keyword might be in the description.
    filtered_news = [
        article for article in all_news
        if keyword.lower() in article['title'].lower() or \
           (article.get('description') and keyword.lower() in article['description'].lower())
    ]

    if top_n is not None:
        filtered_news = filtered_news[:top_n]

    if not filtered_news:
        return f"No news found for '{keyword}'."

    output = f"{keyword.upper()} NEWS\n" + "─" * 20 + "\n"
    for i, article in enumerate(filtered_news):
        output += f"{i + 1}. {article['title']}\n"
    output += "\nRead more → ..."
    return output

### Get Your Personalized News

Use the interactive fields below to enter your desired keyword and specify if you want only the top 5 articles.

In [29]:
#@title Enter your news preferences
keyword_input = "technology" #@param {type:"string"}
top_5_only = True #@param {type:"boolean"}

if top_5_only:
    news_digest = get_personalized_news(keyword_input, top_n=5)
else:
    news_digest = get_personalized_news(keyword_input)

print(news_digest)

Error fetching news from API: {'status': 'error', 'code': 'apiKeyInvalid', 'message': 'Your API key is invalid or incorrect. Check your key, or go to https://newsapi.org to create a free API key.'}
No news found for 'technology'.


In [30]:
# Re-running the news digest generation after checking the API key.

if top_5_only:
    news_digest = get_personalized_news(keyword_input, top_n=5)
else:
    news_digest = get_personalized_news(keyword_input)

print(news_digest)

Error fetching news from API: {'status': 'error', 'code': 'apiKeyInvalid', 'message': 'Your API key is invalid or incorrect. Check your key, or go to https://newsapi.org to create a free API key.'}
No news found for 'technology'.
